# 3. A grateful-patient pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PhilanthroPy-Project/PhilanthroPy/blob/main/examples/notebooks/03_grateful_patient_pipeline.ipynb)

Academic medical centers have a channel most nonprofits do not: patients who
give because of the care they received. This notebook turns clinical encounters
into model-ready features and scores prospects with them.

> **Read this before doing it for real.** The PII handling in these transformers
> is a narrow read surface, **not** formal HIPAA de-identification, and the
> service-line capacity weights are illustrative with no published source. See
> [Compliance considerations](https://philanthropy-project.github.io/PhilanthroPy/explanation/compliance_considerations/).
> Every number below is synthetic; no real patient data appears anywhere in this
> package.

In [ ]:
try:
    from philanthropy.datasets import make_donor_panel
except ImportError:
    !pip install -q "philanthropy @ git+https://github.com/PhilanthroPy-Project/PhilanthroPy@main"
    from philanthropy.datasets import make_donor_panel

import philanthropy
print("philanthropy", philanthropy.__version__)

## Two tables

Encounters join to donors on an identifier that never reaches the model.

In [ ]:
import numpy as np
import pandas as pd

panel = make_donor_panel(
    n_donors=800, n_years=5, include_encounters=True, random_state=7
)
gifts, donors, encounters = panel["gifts"], panel["donors"], panel["encounters"]

print(f"{len(donors)} donors, {len(gifts)} gifts, {len(encounters)} encounters")
encounters.head()

## Encounter dates are drawn independently of giving

Worth saying out loud, because it is what makes the exercise meaningful. A
generator that made grateful-patient features predictive by construction would
be a very convincing demonstration of nothing. Any signal the model finds below
comes from the label being built out of encounter history, not from the
generator quietly correlating the two.

In [ ]:
# One row per donor, with their most recent gift date as the decision point.
X = (
    gifts.groupby("donor_id", as_index=False)["gift_date"].max()
    .merge(donors, on="donor_id", how="right")
)
X["gift_date"] = X["gift_date"].fillna(gifts["gift_date"].max())
# EncounterTransformer currently raises on a real datetime64 gift_date column
# and only accepts date strings (philanthropy#163); this round-trip is the
# workaround, not the recommended pattern.
X["gift_date"] = X["gift_date"].dt.strftime("%Y-%m-%d")

# The label: donors seen three or more times. Synthetic, and deliberately a
# function of the encounter table, so there is something real to recover.
counts = encounters["donor_id"].value_counts()
y = (X["donor_id"].map(counts).fillna(0) >= 3).astype(int).to_numpy()
print("base rate", y.mean().round(3))

## `as_of` is not optional

`EncounterTransformer(as_of=...)` drops every encounter discharged after the
cutoff. Without it you are building today's features out of admissions that had
not happened when the decision was made, which is notebook 2's mistake wearing
a lab coat.

Below, the cutoff is set a year before the end of the data, and the difference
it makes is measured rather than asserted.

In [ ]:
from philanthropy.preprocessing import EncounterTransformer

cutoff = encounters["discharge_date"].max() - pd.Timedelta(days=365)

cols = ["donor_id", "gift_date"]


def encounter_features(as_of):
    t = EncounterTransformer(
        encounter_df=encounters,
        discharge_col="discharge_date",
        gift_date_col="gift_date",
        merge_key="donor_id",
        as_of=as_of,
    )
    t.set_output(transform="pandas")
    return t.fit_transform(X[cols])


with_cutoff = encounter_features(cutoff)
without_cutoff = encounter_features(None)

pd.DataFrame({
    "as_of set": with_cutoff["encounter_frequency_score"].describe(),
    "as_of unset (leaks)": without_cutoff["encounter_frequency_score"].describe(),
}).round(2)

The unset column counts encounters that had not happened yet. On real data that
difference is free accuracy in your backtest and none in production.

## Service-line weighting

`GratefulPatientFeaturizer` reads only encounter metadata: service line,
attending physician, dates. Four numeric aggregates come out and no identifier
does. Capacity weighting is **off by default**, because the built-in multipliers
are illustrative and turning them on silently would launder a guess into a
headline score.

In [ ]:
from philanthropy.preprocessing import GratefulPatientFeaturizer

gpf = GratefulPatientFeaturizer(encounter_df=encounters)
clinical = pd.DataFrame(
    gpf.fit_transform(X[["donor_id"]]), columns=gpf.get_feature_names_out()
)
clinical.head()

## The solicitation window

Patients 90 to 365 days post-discharge are often the warmest prospects: long
enough that the visit is not raw, recent enough that it is not forgotten.
`window_position_score` peaks at the midpoint and falls to zero at either edge.

In [ ]:
from philanthropy.preprocessing import DischargeToSolicitationWindowTransformer

window = DischargeToSolicitationWindowTransformer()
scored = pd.DataFrame(
    window.fit_transform(with_cutoff[["days_since_last_discharge"]]),
    columns=window.get_feature_names_out(),
)
print("in window:", int(scored["in_solicitation_window"].sum()), "of", len(scored))
scored.head()

## Route with `ColumnTransformer`, not a serial `Pipeline`

Each of these transformers consumes named columns and replaces them. Chained in
series, the second receives the first's output under the wrong names, produces a
constant feature block, and trains a model on nothing while exiting 0. A
`ColumnTransformer` hands each branch exactly the columns it wants.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from philanthropy.models import MajorGiftClassifier
from philanthropy.preprocessing import CRMCleaner

pipeline = Pipeline([
    ("features", ColumnTransformer(
        transformers=[
            ("encounters", EncounterTransformer(
                encounter_df=encounters,
                discharge_col="discharge_date",
                gift_date_col="gift_date",
                merge_key="donor_id",
                as_of=cutoff,
            ), ["donor_id", "gift_date"]),
            ("clinical", GratefulPatientFeaturizer(encounter_df=encounters), ["donor_id"]),
            ("crm", CRMCleaner(), ["wealth_estimate"]),
        ],
        remainder="drop",
    )),
    # MajorGiftClassifier handles NaN natively, so the ~30% missing wealth
    # estimates need no upstream imputation.
    ("model", MajorGiftClassifier(max_iter=50, random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)
pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score

proba = pipeline.predict_proba(X_test)[:, 1]
print(f"held-out ROC-AUC: {roc_auc_score(y_test, proba):.3f}")

# A constant score means the features never reached the model: exactly the
# failure the ColumnTransformer prevents. Assert it, do not eyeball it.
assert len(set(proba.round(6))) > 1, "degenerate pipeline: every score identical"

## What to take from this

The pipeline shape matters more than the estimator. `as_of` on the encounter
step, `ColumnTransformer` rather than a serial chain, capacity weights off until
your institution has reviewed them, and a compliance read before any of it
touches a real EHR export.

Then re-validate on your own data. These numbers are synthetic and say nothing
about your program.